**Install Required Libraries**

In [ ]:
!pip install sentence_transformers faiss-cpu pandas -q

**Importing Libraries**

In [ ]:
import pandas as pd
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer

**Upload Dataset**

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving customer_reviews.csv to customer_reviews (1).csv


**Loading Dataset**

In [ ]:
df = pd.read_csv("customer_reviews.csv")
df.head()

,review_id,product_id,review_text,rating
0,1,P101,Excellent battery life and superb camera quali...,5
1,2,P101,"The phone lasts two days on a single charge, g...",5
2,3,P101,Battery is good but the screen is average.,4
3,4,P101,Amazing camera but the design feels outdated.,4
4,5,P101,Very reliable battery performance.,5


**Dataset Overview**

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   review_id    50 non-null     int64 
 1   product_id   50 non-null     object
 2   review_text  50 non-null     object
 3   rating       50 non-null     int64 
dtypes: int64(2), object(2)
memory usage: 1.7+ KB


**Checking Missing Values**

In [ ]:
df.isnull().sum()

,0
review_id,0
product_id,0
review_text,0
rating,0


**Creating Searchable Text**

In [ ]:
df["combined_text"] = (
    "Product ID: " + df["product_id"].astype(str) +
    ".Review Text: " + df["review_text"].astype(str) +
    ".Rating: " + df["rating"].astype(str)
)

df["combined_text"].head()

,combined_text
0,Product ID: P101.Review Text: Excellent batter...
1,Product ID: P101.Review Text: The phone lasts ...
2,Product ID: P101.Review Text: Battery is good ...
3,Product ID: P101.Review Text: Amazing camera b...
4,Product ID: P101.Review Text: Very reliable ba...


**Loading Embedding Model**

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


**Creating Embeddings**

In [ ]:
embeddings = model.encode(
    df["combined_text"].tolist(),
    show_progress_bar=True
)

embeddings.shape

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

(50, 384)

**Converting Embeddings to NumPy Array**

In [ ]:
embedding_array = np.array(embeddings).astype('float32')

**Creating FAISS Index**

In [ ]:
dimension = embedding_array.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embedding_array)

**Search Function**
1. Takes user query.
2. Converts query into vector.
3. Searches similar vectors.
4. Retrieve matching products.

In [ ]:
def search_products(query,top_k=5):

  query_vector = model.encode([query])

  query_vector = np.array(query_vector).astype("float32")

  distances,indices = index.search(query_vector,top_k)

  results = df.iloc[indices[0]][
      ["product_id","review_text","rating"]
  ]

  return results

**Testing Recommendation Search Engine**

In [ ]:
search_products("Best phone for gaming")

,product_id,review_text,rating
22,P105,"Great for basic use, not for gaming.",4
11,P103,"Runs heavy games without lag, excellent speed.",5
29,P106,Best sound system I've used in a phone.,5
7,P102,The phone heats up quickly during use.,1
10,P103,Blazing fast performance and smooth multitasking.,5


In [ ]:
search_products("Phone with excellent battery life")

,product_id,review_text,rating
1,P101,"The phone lasts two days on a single charge, g...",5
7,P102,The phone heats up quickly during use.,1
14,P103,Incredible speed but battery drains fast.,4
29,P106,Best sound system I've used in a phone.,5
2,P101,Battery is good but the screen is average.,4


In [ ]:
search_products("Best phone for travelers")

,product_id,review_text,rating
24,P105,Low cost and works well for calls.,5
29,P106,Best sound system I've used in a phone.,5
7,P102,The phone heats up quickly during use.,1
1,P101,"The phone lasts two days on a single charge, g...",5
15,P104,"Extremely lightweight and portable, great for ...",5


In [ ]:
search_products("fast and smooth multitasking")

,product_id,review_text,rating
10,P103,Blazing fast performance and smooth multitasking.,5
11,P103,"Runs heavy games without lag, excellent speed.",5
5,P102,"Poor performance, lags frequently when opening...",2
12,P103,Very responsive and fast loading times.,5
8,P102,Touch response is slow and frustrating.,2


In [ ]:
search_products("Affordable phone with good value")

,product_id,review_text,rating
29,P106,Best sound system I've used in a phone.,5
24,P105,Low cost and works well for calls.,5
7,P102,The phone heats up quickly during use.,1
21,P105,Budget-friendly but camera is basic.,4
1,P101,"The phone lasts two days on a single charge, g...",5


**Product Recommendation Logic**


In [ ]:
def recommend_product(query,top_k=10):

  query_vector = model.encode([query])

  query_vector = np.array(query_vector).astype("float32")

  distances,indices = index.search(query_vector,top_k)

  results = df.iloc[indices[0]]

  recommendation = (
      results.groupby("product_id")
      .agg(
          average_rating = ("rating","mean"),
          review_count = ("review_id","count")
      )
      .sort_values(
          by=["average_rating","review_count"],
          ascending=False
      )
  )

  return recommendation

In [ ]:
recommend_product("Best phone for gaming")

,average_rating,review_count
product_id,,
P103,5.000000,2
P106,5.000000,1
P105,4.500000,2
P101,4.000000,1
P107,4.000000,1
P102,1.666667,3


In [ ]:
recommend_product("Phone with excellent battery life")

,average_rating,review_count
product_id,,
P105,5.000000,1
P106,5.000000,1
P101,4.666667,3
P109,4.500000,2
P103,4.000000,1
P104,3.000000,1
P102,1.000000,1


In [ ]:
recommend_product("Best phone for travelers")

,average_rating,review_count
product_id,,
P101,5.0,2
P104,5.0,1
P106,5.0,1
P109,5.0,1
P105,4.5,2
P103,4.0,1
P102,1.5,2


**Saving FAISS Index**

In [ ]:
faiss.write_index(index,"product_index.faiss")

**Save Embeddings**

In [ ]:
np.save("product_embeddings.npy",embedding_array)

**Loading Saved FAISS Index**

In [ ]:
loaded_index = faiss.read_index("product_index.faiss")

**Real Time Interactive Search**

In [ ]:
while True:
  user_query = input("Enter your search query:")
  if user_query.lower() == "exit":
    break

  results = recommend_product(user_query)
  print(results)

Enter your search query:Best phone for gaming 
            average_rating  review_count
product_id                              
P103              5.000000             2
P106              5.000000             1
P105              4.500000             2
P101              4.000000             1
P107              4.000000             1
P102              1.666667             3
Enter your search query:Best phone for travelers
            average_rating  review_count
product_id                              
P101                   5.0             2
P104                   5.0             1
P106                   5.0             1
P109                   5.0             1
P105                   4.5             2
P103                   4.0             1
P102                   1.5             2
Enter your search query:Phone with excellent battery life
            average_rating  review_count
product_id                              
P105              5.000000             1
P106              5.000000

# Insights

1. Semantic search understands meaning instead of exact keywords.

2. Products with reviews mentioning:
   - fast performance
   - smooth gaming
   - multitasking
   are recommended for gaming queries.

3. Products with:
   - lightweight
   - portable
   - travel-friendly
   are recommended for travel purposes.

4. Embedding models capture contextual meaning of customer reviews.

5. FAISS enables extremely fast vector similarity search.
